In [32]:
from pylib.setup import *
setup_notebook()

from assumptions import demand, smr_tech, SMR_DOWNTIME, solar_tech, wind_tech, battery, GRID_TARIFF_BUY, GRID_TARIFF_SELL, grid_connect_annual
from model import GridSupply, KKSupply, VESupply, DatacenterModel

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Load inputs
*Three 8760-row txt files from `1_VE.ipynb`. Solar and wind are normalised capacity factors (production / installed capacity). Prices are hourly DK weighted-average spot prices in €/MWh.*

In [33]:
pat = pathlib.Path("variation_patterns")

# Installed capacities from 1_VE.ipynb API output — used to normalise MWh/h → CF [0,1]
_solar_cap_mw = 4_955.5428
_wind_cap_mw  = 4_878.483

solar_cf = np.loadtxt(pat / "PV_VE_2025_2026.txt") / _solar_cap_mw   # MWh/h → CF
wind_cf  = np.loadtxt(pat / "WL_VE_2025_2026.txt") / _wind_cap_mw    # MWh/h → CF
prices   = np.loadtxt(pat / "wp_2025_2026.txt")    / 7.46             # DKK/MWh → EUR/MWh

print(f"solar_cf : {solar_cf.shape}  mean={solar_cf.mean():.3f}  max={solar_cf.max():.3f}")
print(f"wind_cf  : {wind_cf.shape}   mean={wind_cf.mean():.3f}  max={wind_cf.max():.3f}")
print(f"prices   : {prices.shape}    mean={prices.mean():.1f}   min={prices.min():.1f}  max={prices.max():.1f}  €/MWh")

solar_cf : (8760,)  mean=0.103  max=0.750
wind_cf  : (8760,)   mean=0.229  max=0.816
prices   : (8760,)    mean=81.6   min=-30.7  max=583.5  €/MWh


## 2. Demand and grid
*`DatacenterDemand` sets the total load (1 GW flat) and the on-site fraction `x`. `GridSupply` wraps the price series and computes import costs given any on-site dispatch profile.*

In [34]:
grid = GridSupply(prices, demand)

print(f"Total demand   : {demand.demand_mw:,.0f} MW")
print(f"On-site floor  : {demand.floor_mw:,.0f} MW  (x = {demand.x:.0%})")
print(f"Grid cap       : {demand.grid_cap_mw:,.0f} MW")
print(f"Annual demand  : {demand.annual_mwh/1e6:.3f} TWh")

Total demand   : 1,000 MW
On-site floor  : 500 MW  (x = 50%)
Grid cap       : 500 MW
Annual demand  : 8.760 TWh


## 3. KK — nuclear on-site
*Constant output at `floor_mw`. Minimum feasible capacity by construction — no sizing needed. Capacity factor enters only the fuel-cost calculation (annual generation = capacity × CF × 8760).*

In [35]:
kk = KKSupply(smr_tech, demand, prices, downtime_fraction=SMR_DOWNTIME,
              buy_tariff=GRID_TARIFF_BUY,
              grid_connect_annual=grid_connect_annual(demand.demand_mw))

print(f"Installed capacity : {kk.capacity_mw:,.0f} MW")
print(f"Downtime           : {kk.downtime_hours} h/yr  ({SMR_DOWNTIME:.0%} of year)")
print(f"Annual generation  : {kk.capacity_mw * (demand.HOURS - kk.downtime_hours) / 1e6:.3f} TWh")
print(f"Annual on-site cost: {kk.annual_cost() / 1e6:.1f} M€")

Installed capacity : 1,000 MW
Downtime           : 876 h/yr  (10% of year)
Annual generation  : 7.884 TWh
Annual on-site cost: 598.7 M€


## 4. VE — solar + wind + battery on-site
*Single LP over all capacity and dispatch variables simultaneously (HiGHS). `storage_hours` fixes E_batt = storage_hours × P_batt; set to `None` to let E_batt optimise freely. Solves in seconds.*

In [36]:
ve = VESupply(solar_cf, wind_cf, solar_tech, wind_tech, battery, demand,
              prices=prices, storage_hours=4,
              buy_tariff=GRID_TARIFF_BUY, sell_tariff=GRID_TARIFF_SELL,
              grid_connect_annual=grid_connect_annual(demand.grid_cap_mw))

t0 = time.time()
_ = ve.solution   # trigger optimisation
print(f"Optimisation done in {time.time()-t0:.1f} s")
print()
print(f"Solar              : {ve.c_solar:,.1f} MW")
print(f"Wind               : {ve.c_wind:,.1f} MW")
dur = f"{ve.batt_energy_mwh/ve.batt_power_mw:.1f}h" if ve.batt_power_mw > 0 else "—"
print(f"Battery            : {ve.batt_power_mw:,.1f} MW / {ve.batt_energy_mwh:,.1f} MWh  ({dur})")
print(f"Annual on-site cost: {ve.annual_onsite_cost() / 1e6:.1f} M€")

ve.save()
ve.save_lp_arrays()
ve.save_lp_txt()
print("Solution saved → runs/ve_solution.json, runs/ve_lp_arrays.npz, runs/lp_arrays/*.txt")

Optimisation done in 12.0 s

Solar              : 6,173.6 MW
Wind               : 3,338.1 MW
Battery            : 7,133.2 MW / 28,532.8 MWh  (4.0h)
Annual on-site cost: 1044.8 M€
Solution saved → runs/ve_solution.json, runs/ve_lp_arrays.npz, runs/lp_arrays/*.txt


## 5. Full model — KK vs VE
*Adds grid purchase costs to each on-site option and computes total annualised cost and system LCOE.*

In [37]:
model = DatacenterModel(demand, grid, kk, ve)
results = model.run()

for r in results.values():
    print(r)
    print(f"  on-site cost   : {r.annual_onsite_cost/1e6:.1f} M€/yr")
    print(f"  grid cost      : {r.annual_grid_cost/1e6:.1f} M€/yr")
    if r.annual_export_revenue > 0:
        print(f"  export revenue : {r.annual_export_revenue/1e6:.1f} M€/yr")
    print(f"  grid imports   : {r.grid_import_mwh/1e6:.3f} TWh/yr")
    print()

KK [1000 MW SMR] | LCOE 75.39 €/MWh | total 660.4 M€/yr
  on-site cost   : 598.7 M€/yr
  grid cost      : 47.9 M€/yr
  grid imports   : 0.876 TWh/yr

VE [6174 MW PV + 3338 MW wind + 7133 MW / 28533 MWh batt] | LCOE 98.86 €/MWh | total 866.0 M€/yr
  on-site cost   : 1044.8 M€/yr
  grid cost      : 63.9 M€/yr
  export revenue : 258.2 M€/yr
  grid imports   : 0.794 TWh/yr

